In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings

warnings.filterwarnings('ignore')

# --- Step 1: Target Encoding & Pure Pandas Preprocessing ---

train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

# Encode Target
le = LabelEncoder()
y = le.fit_transform(train_df['class'])

# Cyclical Trigonometry for 'alpha'
def process_features(df):
    df = df.copy()
    df['alpha_sin'] = np.sin(df['alpha'] * np.pi / 180)
    df['alpha_cos'] = np.cos(df['alpha'] * np.pi / 180)

    # Spectral Type Mapping
    spec_map = {'M': 0, 'G/K': 1, 'A/F': 2, 'O/B': 3}
    df['spectral_type'] = df['spectral_type'].map(spec_map).astype('int')

    # Galaxy Population Encoding
    gp_le = LabelEncoder()
    df['galaxy_population'] = gp_le.fit_transform(df['galaxy_population']).astype('int')

    # Drop redundant columns
    return df.drop(columns=['id', 'alpha', 'class'], errors='ignore')

X_full = process_features(train_df)
X_test = process_features(test_df)

print(f'Preprocessing complete. Feature count: {X_full.shape[1]}')

In [ ]:
# --- Step 2: Phase 1 - High-Speed Optuna Scout (GPU Accelerated) ---

def lgb_objective(trial):
    params = {
        'device': 'gpu',
        'class_weight': 'balanced',
        'verbose': -1,
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'num_leaves': trial.suggest_int('num_leaves', 16, 96),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in skf.split(X_full, y):
        model = LGBMClassifier(**params)
        model.fit(X_full.iloc[train_idx], y[train_idx])
        scores.append(accuracy_score(y[val_idx], model.predict(X_full.iloc[val_idx])))
    return np.mean(scores)

def cat_objective(trial):
    params = {
        'task_type': 'GPU',
        'auto_class_weights': 'Balanced',
        'verbose': False,
        'iterations': trial.suggest_int('iterations', 300, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 20.0, log=True),
        'random_state': 42
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in skf.split(X_full, y):
        model = CatBoostClassifier(**params)
        model.fit(X_full.iloc[train_idx], y[train_idx], cat_features=['spectral_type', 'galaxy_population'])
        scores.append(accuracy_score(y[val_idx], model.predict(X_full.iloc[val_idx])))
    return np.mean(scores)

print("Starting Optuna for LightGBM...")
lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_objective, n_trials=10)

print("Starting Optuna for CatBoost...")
cat_study = optuna.create_study(direction='maximize')
cat_study.optimize(cat_objective, n_trials=10)

lgb_best_params = lgb_study.best_params
cat_best_params = cat_study.best_params

In [ ]:
# --- Step 3: Phase 2 - The Variance Shield (5-Fold OOF Loop) ---

n_classes = len(np.unique(y))
lgb_oof = np.zeros((len(X_full), n_classes))
cat_oof = np.zeros((len(X_full), n_classes))
lgb_test_preds = np.zeros((len(X_test), n_classes))
cat_test_preds = np.zeros((len(X_test), n_classes))

skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (t_idx, v_idx) in enumerate(skf_final.split(X_full, y)):
    print(f"Training Fold {fold+1}...")
    X_tr, X_val = X_full.iloc[t_idx], X_full.iloc[v_idx]
    y_tr, y_val = y[t_idx], y[v_idx]

    # LightGBM - GPU Accelerated with optimized params
    lgb_model = LGBMClassifier(**lgb_best_params, device='gpu', class_weight='balanced', verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
    lgb_oof[v_idx] = lgb_model.predict_proba(X_val)
    lgb_test_preds += lgb_model.predict_proba(X_test) / 5

    # CatBoost - GPU Accelerated with optimized params and categorical handling
    cat_model = CatBoostClassifier(**cat_best_params, task_type='GPU', auto_class_weights='Balanced', verbose=False)
    cat_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], cat_features=['spectral_type', 'galaxy_population'], early_stopping_rounds=50)
    cat_oof[v_idx] = cat_model.predict_proba(X_val)
    cat_test_preds += cat_model.predict_proba(X_test) / 5

In [ ]:
# --- Step 4: Final Ensembling & Submission ---

meta_X_train = np.hstack([lgb_oof, cat_oof])
meta_X_test = np.hstack([lgb_test_preds, cat_test_preds])

meta_model = LogisticRegression(max_iter=1000, random_state=42)
meta_model.fit(meta_X_train, y)

final_oof_preds = meta_model.predict(meta_X_train)
final_accuracy = accuracy_score(y, final_oof_preds)
print(f"Final Stacking OOF Accuracy: {final_accuracy:.5f}")

# Generate Submission
final_test_preds = meta_model.predict(meta_X_test)
submission = pd.DataFrame({
    'id': test_df['id'],
    'class': le.inverse_transform(final_test_preds)
})
submission.to_csv('l2_regularized_update.csv', index=False)
print("Submission file saved as submission.csv")